# ViT-Ctr Phase 3: 模型评估
从 `checkpoints/best_model.pth` 加载训练好的模型，在测试集上生成全套评估指标和图表。

In [1]:
import sys, os
sys.path.insert(0, '../src')

In [ ]:
# 配置
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_PATH = '../checkpoints/best_model.pth'
H5_PATHS = [
    '../data/dithioester.h5',
    '../data/trithiocarbonate.h5',
    '../data/xanthate.h5',
    '../data/dithiocarbamate.h5',
]
FIGURES_DIR = '../figures'
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

In [1]:
# 加载模型
from model import SimpViT
model = SimpViT(num_outputs=3).to(DEVICE)
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.6f}")
else:
    print(f"[WARN] Checkpoint not found: {CHECKPOINT_PATH}")
    print("请先完成训练（colab/03-train-colab.ipynb），然后将 best_model.pth 放到 checkpoints/")
model.eval()

ModuleNotFoundError: No module named 'model'

In [ ]:
# 构建测试集DataLoader
from dataset import CombinedHDF5Dataset
from utils.split import build_stratified_indices
from torch.utils.data import DataLoader

train_idx, val_idx, test_idx = build_stratified_indices(H5_PATHS, seed=42)
test_ds = CombinedHDF5Dataset(H5_PATHS, test_idx)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)
print(f"测试集样本数: {len(test_ds)}")

In [ ]:
# 运行完整评估
from evaluate import run_full_evaluation
results = run_full_evaluation(model, test_loader, DEVICE, figures_dir=FIGURES_DIR)

In [ ]:
# 指标汇总表
import pandas as pd
from evaluate import OUTPUT_NAMES

rows = []
for i, name in enumerate(OUTPUT_NAMES):
    rows.append({
        'Output': name,
        'R²': f"{results['overall']['r2'][i]:.4f}",
        'RMSE': f"{results['overall']['rmse'][i]:.4f}",
        'MAE': f"{results['overall']['mae'][i]:.4f}",
    })
pd.DataFrame(rows)

In [ ]:
# 按Ctr范围分段指标
for seg, metrics in results['segmented'].items():
    print(f"\n--- {seg} range ---")
    for i, name in enumerate(OUTPUT_NAMES):
        print(f"  {name}: R²={metrics['r2'][i]:.4f}, RMSE={metrics['rmse'][i]:.4f}")

In [ ]:
# 按RAFT类型指标
for raft_type, metrics in results['by_class'].items():
    print(f"\n--- {raft_type} ---")
    for i, name in enumerate(OUTPUT_NAMES):
        print(f"  {name}: R²={metrics['r2'][i]:.4f}")

## 图表
所有parity图和残差图已保存到 `figures/` 目录。
- `parity_*.png`: 总体parity图（3张）
- `parity_by_class/*.png`: 按RAFT类型parity图（12张）
- `residuals_*.png`: 残差直方图（3张）

**注**: retardation_factor 对 TTC/xanthate/dithiocarbamate 的 R² 接近 0 是预期行为，反映了这些体系的减速因子在物理上恒为 ~1.0。